# Target Single-Best Ensemble From Bundles

This notebook builds one submission by selecting a single best source for each `(model_id, node_type)` target.

Overview:
- discover compatible prediction bundles from the configured roots
- recompute an OOF LB-like score for each source and each target directly from `oof_predictions`
- choose the single best source for each target and apply it to all test events in that target
- report per-target selected-source summaries and the overall mean across targets
- write submission, summary tables, and test-event choice tables

Use this notebook when you want a simple target-wise baseline ensemble without per-event or per-node selection.


In [1]:
import os
import gc
import json
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd


SCHEMA_VERSION = "ufb_ensemble_v1"
STD_DEV = {
    (1, 1): 16.877747,
    (1, 2): 14.378797,
    (2, 1): 3.191784,
    (2, 2): 2.727131,
}

def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_path_list(raw: str) -> List[Path]:
    out: List[Path] = []
    for tok in raw.split(","):
        tok = tok.strip()
        if not tok:
            continue
        out.append(Path(tok))
    return out


def parse_targets(raw: str) -> List[Tuple[int, int]]:
    out: List[Tuple[int, int]] = []
    for tok in raw.split(","):
        tok = tok.strip()
        if not tok:
            continue
        sp = tok.split(":")
        if len(sp) != 2:
            raise ValueError(f"invalid TARGETS token: {tok}")
        out.append((int(sp[0]), int(sp[1])))
    if not out:
        raise ValueError("TARGETS is empty")
    return out


def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


# ---------------------------
# Config
# ---------------------------
BUNDLE_ROOTS = parse_path_list(
    os.environ.get(
        "BUNDLE_ROOTS",
        "/kaggle/input,/kaggle/working/model_bundles,ensemble/artifacts/model_bundles",
    )
)
MANIFEST_NAME = os.environ.get("MANIFEST_NAME", "bundle_manifest.json")
STRICT_REQUIRE_MANIFESTS = env_bool("STRICT_REQUIRE_MANIFESTS", False)
TARGETS = parse_targets(os.environ.get("TARGETS", "1:1,1:2,2:1,2:2"))

EXPECTED_FOLDS = env_int("EXPECTED_FOLDS", 5)
EXPECTED_SEED = env_int("EXPECTED_SEED", 2026)
ALLOW_FOLD_MISMATCH = env_bool("ALLOW_FOLD_MISMATCH", True)

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/kaggle/input/datasets/mtmrs1/urban-flood-bench-re"))
MODELS_ROOT = Path(os.environ.get("MODELS_ROOT", "")) if os.environ.get("MODELS_ROOT", "").strip() else None
SAMPLE_SUBMISSION_PATH = os.environ.get(
    "SAMPLE_SUBMISSION_PATH",
    "/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv",
)

RAIN_ALIGN_MODE = os.environ.get("RAIN_ALIGN_MODE", "min").strip().lower()  # min|fixed
RAIN_ALIGN_FIXED_LEN = env_int("RAIN_ALIGN_FIXED_LEN", 0)
RAIN_MAX_FEATURES = env_int("RAIN_MAX_FEATURES", 0)

OOF_CSV_CHUNK_ROWS = max(100_000, env_int("OOF_CSV_CHUNK_ROWS", 1_000_000))
OOF_PARQUET_BATCH_ROWS = max(100_000, env_int("OOF_PARQUET_BATCH_ROWS", 1_000_000))
TEST_CSV_CHUNK_ROWS = max(100_000, env_int("TEST_CSV_CHUNK_ROWS", 1_000_000))
TEST_PARQUET_BATCH_ROWS = max(100_000, env_int("TEST_PARQUET_BATCH_ROWS", 1_000_000))
AGG_FLUSH_GROUP_ROWS = max(200_000, env_int("AGG_FLUSH_GROUP_ROWS", 2_000_000))
MIN_OOF_TIMESTEPS = max(1, env_int("MIN_OOF_TIMESTEPS", 1))

OUTPUT_DIR = Path(
    os.environ.get(
        "OUTPUT_DIR",
        "/kaggle/working/ensemble_output_target_singlebest_cv"
        if Path("/kaggle/working").exists()
        else "ensemble/artifacts/ensemble_runs/target_singlebest_cv",
    )
)
PUBLIC_SUBMISSION_PATH = os.environ.get(
    "PUBLIC_SUBMISSION_PATH",
    "/kaggle/working/submission.csv" if Path("/kaggle/working").exists() else "",
).strip()

CV_SCORE_COL_PRIORITY = [
    c.strip()
    for c in os.environ.get(
        "CV_SCORE_COL_PRIORITY",
        "oof_lb_std,std_rmse,sample_std,mean_std_rmse,selector_cv_std_rmse,single_best_std_rmse",
    ).split(",")
    if c.strip()
]
CV_ALLOW_OOF_FALLBACK = env_bool("CV_ALLOW_OOF_FALLBACK", True)


# ---------------------------
# Bundle discovery
# ---------------------------
@dataclass
class Bundle:
    name: str
    source_kind: str
    n_folds: int
    event_split_seed: int
    oof_path: Path
    test_path: Path
    cv_path: Path | None = None
    manifest_path: Path | None = None


def _required(raw: Dict, key: str):
    if key not in raw:
        raise KeyError(f"manifest missing required key: {key}")
    return raw[key]


def _resolve_rel(manifest_path: Path, rel_or_abs: str) -> Path:
    p = Path(rel_or_abs)
    if p.is_absolute():
        return p
    return (manifest_path.parent / p).resolve()


def discover_manifest_paths(roots: Iterable[Path], manifest_name: str) -> List[Path]:
    out: List[Path] = []
    for root in roots:
        if not root.exists():
            continue
        out.extend(sorted(root.rglob(manifest_name)))
    return out


def load_manifest_bundle(manifest_path: Path) -> Bundle:
    raw = json.loads(manifest_path.read_text())
    schema = str(_required(raw, "schema_version"))
    if schema != SCHEMA_VERSION:
        raise ValueError(f"{manifest_path}: schema_version mismatch: {schema}")

    name = str(_required(raw, "bundle_name")).strip()
    if not name:
        raise ValueError(f"{manifest_path}: empty bundle_name")

    n_folds = int(_required(raw, "n_folds"))
    seed = int(_required(raw, "event_split_seed"))
    oof_path = _resolve_rel(manifest_path, str(_required(raw, "oof_predictions_path")))
    test_path = _resolve_rel(manifest_path, str(_required(raw, "test_predictions_path")))
    cv_path = None
    if "cv_summary_path" in raw:
        try:
            cand = _resolve_rel(manifest_path, str(raw["cv_summary_path"]))
            if cand.exists():
                cv_path = cand
        except Exception:
            cv_path = None

    if not oof_path.exists():
        raise FileNotFoundError(f"{manifest_path}: missing oof file: {oof_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"{manifest_path}: missing test file: {test_path}")

    return Bundle(
        name=name,
        source_kind="manifest",
        n_folds=n_folds,
        event_split_seed=seed,
        oof_path=oof_path,
        test_path=test_path,
        cv_path=cv_path,
        manifest_path=manifest_path,
    )


def discover_loose_bundles(roots: Iterable[Path]) -> List[Bundle]:
    dir_map: Dict[Path, Dict[str, List[Path]]] = {}

    def mark(parent: Path, kind: str, p: Path):
        rec = dir_map.setdefault(parent, {"oof": [], "test": []})
        rec[kind].append(p)

    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            if p.suffix.lower() not in {".csv", ".parquet", ".pq"}:
                continue
            lower = p.name.lower()
            if "oof" in lower and ("pred" in lower or "prediction" in lower):
                mark(p.parent, "oof", p)
            elif "test" in lower and ("pred" in lower or "prediction" in lower):
                mark(p.parent, "test", p)

    out: List[Bundle] = []
    used = set()
    for d, rec in sorted(dir_map.items(), key=lambda x: str(x[0])):
        if not rec["oof"] or not rec["test"]:
            continue
        name = f"loose__{d.name}"
        base = name
        k = 2
        while name in used:
            name = f"{base}_{k}"
            k += 1
        used.add(name)
        out.append(
            Bundle(
                name=name,
                source_kind="loose",
                n_folds=-1,
                event_split_seed=-1,
                oof_path=sorted(rec["oof"])[0],
                test_path=sorted(rec["test"])[0],
                cv_path=None,
                manifest_path=None,
            )
        )
    return out


def _parse_target_from_text(x: str) -> Tuple[int, int] | None:
    s = str(x).strip()
    if not s:
        return None
    s = s.replace("(", "").replace(")", "").replace(" ", "")
    s = s.replace(",", ":")
    parts = s.split(":")
    if len(parts) != 2:
        return None
    try:
        return int(parts[0]), int(parts[1])
    except Exception:
        return None


def read_target_score_from_cv_summary(
    cv_path: Path,
    target: Tuple[int, int],
) -> Tuple[float | None, str | None]:
    suffix = cv_path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(cv_path, low_memory=False)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(cv_path)
    else:
        return None, None

    if len(df) == 0:
        return None, None

    model_id, node_type = int(target[0]), int(target[1])
    picked = None

    if "model_id" in df.columns and "node_type" in df.columns:
        mid = pd.to_numeric(df["model_id"], errors="coerce")
        nty = pd.to_numeric(df["node_type"], errors="coerce")
        m = (mid == model_id) & (nty == node_type)
        if m.any():
            picked = df.loc[m].copy()

    if picked is None and "target" in df.columns:
        parsed = df["target"].map(_parse_target_from_text)
        m = parsed == (model_id, node_type)
        if m.any():
            picked = df.loc[m].copy()

    if picked is None or len(picked) == 0:
        return None, None

    for col in CV_SCORE_COL_PRIORITY:
        if col not in picked.columns:
            continue
        vals = pd.to_numeric(picked[col], errors="coerce").to_numpy(dtype=np.float64, copy=False)
        vals = vals[np.isfinite(vals)]
        if vals.size > 0:
            return float(np.mean(vals)), col

    return None, None


# ---------------------------
# IO helpers
# ---------------------------
OOF_USECOLS = [
    "model_id",
    "event_id",
    "node_type",
    "water_level_true",
    "water_level_pred",
]
TEST_USECOLS_BASE = [
    "row_id",
    "model_id",
    "event_id",
    "node_type",
    "node_id",
    "water_level_pred",
]


def _dtype_map_for_columns(cols: List[str]) -> Dict[str, str]:
    base = {
        "row_id": "int64",
        "model_id": "int16",
        "event_id": "int32",
        "node_type": "int16",
        "node_id": "int32",
        "water_level_true": "float32",
        "water_level_pred": "float32",
    }
    return {c: t for c, t in base.items() if c in cols}


def iter_oof_chunks_for_target(path: Path, target: Tuple[int, int]):
    model_id, node_type = int(target[0]), int(target[1])
    suffix = path.suffix.lower()

    if suffix == ".csv":
        header = pd.read_csv(path, nrows=0)
        cols = [c for c in OOF_USECOLS if c in header.columns]
        dtypes = _dtype_map_for_columns(cols)
        reader = pd.read_csv(
            path,
            usecols=cols,
            dtype=dtypes if dtypes else None,
            chunksize=OOF_CSV_CHUNK_ROWS,
            low_memory=False,
        )
        for chunk in reader:
            if "model_id" not in chunk.columns or "node_type" not in chunk.columns:
                continue
            mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
            mask = (mid == model_id) & (nty == node_type)
            if not np.any(mask):
                continue
            out = chunk.loc[mask, [c for c in OOF_USECOLS if c in chunk.columns]].copy()
            yield out
        return

    if suffix in {".parquet", ".pq"}:
        try:
            import pyarrow.parquet as pq

            pf = pq.ParquetFile(path)
            for batch in pf.iter_batches(columns=OOF_USECOLS, batch_size=OOF_PARQUET_BATCH_ROWS):
                chunk = batch.to_pandas()
                mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
                nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
                mask = (mid == model_id) & (nty == node_type)
                if not np.any(mask):
                    continue
                out = chunk.loc[mask, [c for c in OOF_USECOLS if c in chunk.columns]].copy()
                yield out
            return
        except Exception as exc:
            log(f"warn: parquet batch read failed ({path.name}): {exc}; fallback pandas")
            df = pd.read_parquet(path, columns=OOF_USECOLS)
            mid = df["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = df["node_type"].to_numpy(dtype=np.int16, copy=False)
            mask = (mid == model_id) & (nty == node_type)
            if np.any(mask):
                yield df.loc[mask, [c for c in OOF_USECOLS if c in df.columns]].copy()
            return

    raise ValueError(f"unsupported oof format: {path}")


def iter_test_chunks_for_target(path: Path, target: Tuple[int, int]):
    model_id, node_type = int(target[0]), int(target[1])
    suffix = path.suffix.lower()

    if suffix == ".csv":
        header = pd.read_csv(path, nrows=0)
        cols = [c for c in TEST_USECOLS_BASE if c in header.columns]
        dtypes = _dtype_map_for_columns(cols)
        reader = pd.read_csv(
            path,
            usecols=cols,
            dtype=dtypes if dtypes else None,
            chunksize=TEST_CSV_CHUNK_ROWS,
            low_memory=False,
        )
        for chunk in reader:
            if "model_id" not in chunk.columns or "node_type" not in chunk.columns:
                continue
            mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
            mask = (mid == model_id) & (nty == node_type)
            if not np.any(mask):
                continue
            out = chunk.loc[mask, [c for c in cols if c in chunk.columns]].copy()
            yield out
        return

    if suffix in {".parquet", ".pq"}:
        try:
            import pyarrow.parquet as pq

            pf = pq.ParquetFile(path)
            cols = TEST_USECOLS_BASE
            for batch in pf.iter_batches(columns=cols, batch_size=TEST_PARQUET_BATCH_ROWS):
                chunk = batch.to_pandas()
                mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
                nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
                mask = (mid == model_id) & (nty == node_type)
                if not np.any(mask):
                    continue
                out = chunk.loc[mask, [c for c in cols if c in chunk.columns]].copy()
                yield out
            return
        except Exception as exc:
            log(f"warn: parquet batch read failed ({path.name}): {exc}; fallback pandas")
            df = pd.read_parquet(path, columns=TEST_USECOLS_BASE)
            mid = df["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = df["node_type"].to_numpy(dtype=np.int16, copy=False)
            mask = (mid == model_id) & (nty == node_type)
            if np.any(mask):
                yield df.loc[mask, [c for c in TEST_USECOLS_BASE if c in df.columns]].copy()
            return

    raise ValueError(f"unsupported test format: {path}")


# ---------------------------
# Dataset helpers
# ---------------------------
def resolve_models_root(dataset_root: Path, explicit: Path | None) -> Path:
    cands: List[Path] = []
    if explicit is not None:
        cands.append(explicit)
    cands.extend(
        [
            dataset_root,
            dataset_root / "Models",
            Path("/kaggle/input/datasets/mtmrs1/urban-flood-bench-re"),
            Path("/kaggle/input/datasets/mtmrs1/urban-flood-bench-re/Models"),
            Path("/kaggle/input/datasets/statfan/urban-flood-data"),
            Path("/kaggle/input/datasets/statfan/urban-flood-data/Models"),
            Path("Models"),
        ]
    )

    uniq: List[Path] = []
    seen = set()
    for c in cands:
        if c in seen:
            continue
        seen.add(c)
        uniq.append(c)

    for c in uniq:
        if (c / "Model_1").exists() and (c / "Model_2").exists():
            return c
        if (c / "Models" / "Model_1").exists() and (c / "Models" / "Model_2").exists():
            return c / "Models"

    checked = ", ".join(str(x) for x in uniq[:16])
    raise RuntimeError(f"Models root not found. checked={checked}")


def model_dir(models_root: Path, model_id: int) -> Path:
    p = models_root / f"Model_{model_id}"
    if not p.exists():
        raise FileNotFoundError(f"missing model dir: {p}")
    return p


def list_event_ids(mdir: Path, split: str) -> List[int]:
    d = mdir / split
    out = []
    if not d.exists():
        return out
    for p in sorted(d.glob("event_*")):
        if not p.is_dir():
            continue
        name = p.name
        try:
            out.append(int(name.split("_")[1]))
        except Exception:
            continue
    return out


def detect_columns(dynamic_path: Path, require_rain: bool = False) -> Tuple[str, str, str, str | None]:
    cols = pd.read_csv(dynamic_path, nrows=0).columns.tolist()

    def pick(cands: List[str], label: str) -> str:
        for c in cands:
            if c in cols:
                return c
        raise RuntimeError(f"{dynamic_path}: {label} column not found; cols={cols[:20]}")

    node_col = pick(["node_id", "node_idx", "node", "id"], "node")
    time_col = pick(["timestep", "t", "time_step", "step"], "time")
    wl_col = pick(["water_level", "waterlevel", "wl"], "water_level")
    rain_col = None
    for c in ["rain", "rainfall", "rain_intensity", "rain_rate"]:
        if c in cols:
            rain_col = c
            break
    if require_rain and rain_col is None:
        raise RuntimeError(f"{dynamic_path}: rain column not found; cols={cols[:20]}")
    return node_col, time_col, wl_col, rain_col


def load_event_rain_series(
    mdir: Path,
    split: str,
    event_id: int,
    node_type: int,
) -> Dict:
    dyn_name = "1d_nodes_dynamic_all.csv" if node_type == 1 else "2d_nodes_dynamic_all.csv"
    path = mdir / split / f"event_{event_id}" / dyn_name
    if not path.exists():
        raise FileNotFoundError(f"missing dynamic file: {path}")

    node_col, time_col, _, rain_col = detect_columns(path, require_rain=False)

    if rain_col is not None:
        rain_use_path = path
        rain_time_col = time_col
        rain_col_use = rain_col
    else:
        # 1D dynamic tables may not contain rain; use paired 2D dynamic file.
        rain_use_path = mdir / split / f"event_{event_id}" / "2d_nodes_dynamic_all.csv"
        if not rain_use_path.exists():
            log(f"warn: rain source not found for event={event_id}; use zeros")
            return {"event_id": int(event_id), "rain": np.zeros((1,), dtype=np.float32)}
        _, rain_time_col, _, rain_col_use = detect_columns(rain_use_path, require_rain=True)

    rain_parts = []
    rcols = [rain_time_col, rain_col_use]
    rreader = pd.read_csv(rain_use_path, usecols=rcols, chunksize=1_000_000, low_memory=False)
    for chunk in rreader:
        rain_parts.append(chunk[[rain_time_col, rain_col_use]])

    if rain_parts:
        rain_df = pd.concat(rain_parts, ignore_index=True)
        rain_df = rain_df.drop_duplicates(subset=[rain_time_col], keep="first").sort_values(rain_time_col)
        rain = rain_df[rain_col_use].to_numpy(dtype=np.float32, copy=False)
        if rain.shape[0] == 0:
            rain = np.zeros((1,), dtype=np.float32)
    else:
        rain = np.zeros((1,), dtype=np.float32)

    return {
        "event_id": int(event_id),
        "rain": rain,
    }


def split_events_kfold(event_ids: List[int], n_folds: int, seed: int) -> List[List[int]]:
    ids = np.asarray(sorted(event_ids), dtype=np.int32)
    if ids.size < 2:
        return [ids.tolist()]
    k = int(max(2, min(n_folds, ids.size)))
    rs = np.random.RandomState(seed)
    perm = rs.permutation(ids)
    folds = [[] for _ in range(k)]
    for i, ev in enumerate(perm.tolist()):
        folds[i % k].append(int(ev))
    for f in folds:
        f.sort()
    return folds


def encode_event_node_key(event_ids: np.ndarray, node_ids: np.ndarray) -> np.ndarray:
    ev = np.asarray(event_ids, dtype=np.int64)
    nd = np.asarray(node_ids, dtype=np.int64)
    return (ev << 32) | (nd & 0xFFFFFFFF)


def build_event_feature_matrix(
    event_summaries: Dict[int, Dict],
    event_ids: List[int],
    rain_len: int,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    n_events = len(event_ids)
    feat_names = [f"rain_t{t}" for t in range(rain_len)]

    x = np.empty((n_events, rain_len), dtype=np.float32)
    ev_out = np.empty((n_events,), dtype=np.int32)

    for i, ev in enumerate(event_ids):
        ss = event_summaries[int(ev)]
        rain = np.asarray(ss["rain"], dtype=np.float32)
        rain_trunc = rain[:rain_len]
        if rain_trunc.shape[0] < rain_len:
            pad = np.zeros((rain_len - rain_trunc.shape[0],), dtype=np.float32)
            rain_trunc = np.concatenate([rain_trunc, pad])

        x[i, :] = rain_trunc
        ev_out[i] = int(ev)

        if (i + 1) % max(1, n_events // 5) == 0 or (i + 1) == n_events:
            log(f"feature build progress: {i + 1}/{n_events} events")

    return x, ev_out, feat_names


# ---------------------------
# OOF aggregation for source selection
# ---------------------------
def aggregate_oof_event_rmse(
    path: Path,
    source_name: str,
    target: Tuple[int, int],
) -> pd.DataFrame:
    parts: List[pd.DataFrame] = []
    acc_rows = 0
    n_chunks = 0

    for chunk in iter_oof_chunks_for_target(path, target):
        n_chunks += 1
        use = chunk[["event_id", "water_level_true", "water_level_pred"]].copy()
        a_true = use["water_level_true"].to_numpy(dtype=np.float32, copy=False)
        a_pred = use["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        keep = np.isfinite(a_true) & np.isfinite(a_pred)
        if not np.any(keep):
            continue
        use = use.loc[keep]

        err = (
            use["water_level_pred"].to_numpy(dtype=np.float64, copy=False)
            - use["water_level_true"].to_numpy(dtype=np.float64, copy=False)
        )
        use["sse"] = (err * err).astype(np.float64, copy=False)

        grp = (
            use.groupby(["event_id"], as_index=False)
            .agg(sse=("sse", "sum"), cnt=("sse", "size"))
            .astype({"event_id": "int32", "sse": "float64", "cnt": "int32"})
        )
        parts.append(grp)
        acc_rows += len(grp)

        if n_chunks % 10 == 0:
            log(
                f"oof agg progress source={source_name} target={target} "
                f"chunks={n_chunks} grouped_rows={acc_rows}"
            )

        if acc_rows >= AGG_FLUSH_GROUP_ROWS:
            tmp = pd.concat(parts, ignore_index=True)
            tmp = (
                tmp.groupby(["event_id"], as_index=False)
                .agg(sse=("sse", "sum"), cnt=("cnt", "sum"))
                .astype({"event_id": "int32", "sse": "float64", "cnt": "int32"})
            )
            parts = [tmp]
            acc_rows = len(tmp)
            gc.collect()

    if not parts:
        return pd.DataFrame(columns=["event_id", "rmse_std", "cnt", "source_name"])

    allg = pd.concat(parts, ignore_index=True)
    allg = (
        allg.groupby(["event_id"], as_index=False)
        .agg(sse=("sse", "sum"), cnt=("cnt", "sum"))
        .astype({"event_id": "int32", "sse": "float64", "cnt": "int32"})
    )

    allg = allg.loc[allg["cnt"] >= int(MIN_OOF_TIMESTEPS)].reset_index(drop=True)
    if len(allg) == 0:
        return pd.DataFrame(columns=["event_id", "rmse_std", "cnt", "source_name"])

    std = float(STD_DEV[(int(target[0]), int(target[1]))])
    allg["rmse_std"] = np.sqrt(allg["sse"] / allg["cnt"].clip(lower=1).to_numpy(dtype=np.float64)) / std
    allg["source_name"] = source_name
    return allg[["event_id", "rmse_std", "cnt", "source_name"]]


# ---------------------------
# Selection helpers
# ---------------------------
def evaluate_choice_with_fallback(
    keys: np.ndarray,
    pred_src_idx: np.ndarray,
    rmse_pivot: pd.DataFrame,
    fallback_order: List[int],
) -> float:
    rows = rmse_pivot.reindex(keys)
    arr = rows.to_numpy(dtype=np.float64, copy=False)
    n = arr.shape[0]
    out = np.full((n,), np.nan, dtype=np.float64)

    ok_pred = (pred_src_idx >= 0) & (pred_src_idx < arr.shape[1])
    idx = np.where(ok_pred)[0]
    if idx.size > 0:
        out[idx] = arr[idx, pred_src_idx[idx]]

    for fi in fallback_order:
        m = np.isnan(out)
        if not np.any(m):
            break
        cand = arr[m, fi]
        good = np.isfinite(cand)
        if np.any(good):
            mm = np.where(m)[0][good]
            out[mm] = cand[good]

    m = np.isnan(out)
    if np.any(m):
        with np.errstate(invalid="ignore"):
            row_min = np.nanmin(arr[m], axis=1)
        good = np.isfinite(row_min)
        if np.any(good):
            mm = np.where(m)[0][good]
            out[mm] = row_min[good]

    return float(np.nanmean(out)) if np.isfinite(out).any() else float("nan")


def map_event_to_choice(event_ids: np.ndarray, sorted_event_ids: np.ndarray, sorted_choice: np.ndarray) -> np.ndarray:
    pos = np.searchsorted(sorted_event_ids, event_ids)
    out = np.full((event_ids.shape[0],), -1, dtype=np.int16)
    valid = (pos >= 0) & (pos < sorted_event_ids.shape[0])
    if np.any(valid):
        pv = pos[valid]
        ev = event_ids[valid]
        good = sorted_event_ids[pv] == ev
        if np.any(good):
            vv = np.where(valid)[0][good]
            out[vv] = sorted_choice[pv[good]]
    return out


def as_float32_array(x) -> np.ndarray:
    if isinstance(x, np.ndarray):
        return x.astype(np.float32, copy=False)
    return np.asarray(x, dtype=np.float32)


# ---------------------------
# Main
# ---------------------------
def main():
    start_all = time.time()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    log(
        f"CONFIG targets={TARGETS} bundle_roots={BUNDLE_ROOTS} "
        f"rain_align_mode={RAIN_ALIGN_MODE}"
    )

    # 1) Discover bundles
    man_paths = discover_manifest_paths(BUNDLE_ROOTS, MANIFEST_NAME)
    bundles: List[Bundle] = []
    for mp in man_paths:
        try:
            b = load_manifest_bundle(mp)
            if b.n_folds != EXPECTED_FOLDS or b.event_split_seed != EXPECTED_SEED:
                if not ALLOW_FOLD_MISMATCH:
                    continue
            bundles.append(b)
        except Exception as exc:
            log(f"manifest skip: {mp} ({exc})")

    if not bundles and not STRICT_REQUIRE_MANIFESTS:
        bundles = discover_loose_bundles(BUNDLE_ROOTS)

    if not bundles:
        raise RuntimeError("no bundles found")

    uniq = {}
    for b in bundles:
        uniq[b.name] = b
    bundles = [uniq[k] for k in sorted(uniq.keys())]
    bundle_by_name = {b.name: b for b in bundles}
    log(f"bundles detected: {len(bundles)} names={[b.name for b in bundles]}")

    # 2) Load sample submission keys
    sample_path = Path(SAMPLE_SUBMISSION_PATH)
    if not sample_path.exists():
        raise RuntimeError(f"sample submission not found: {sample_path}")

    required_cols = ["row_id", "model_id", "event_id", "node_type", "node_id"]
    header_cols = pd.read_csv(sample_path, nrows=0).columns.tolist()
    miss = [c for c in required_cols if c not in header_cols]
    if miss:
        raise RuntimeError(
            "sample_submission missing required columns for this notebook: "
            f"missing={miss} path={sample_path}"
        )

    sample = pd.read_csv(sample_path, low_memory=False)
    cast_map = {
        "row_id": "int64",
        "model_id": "int16",
        "event_id": "int32",
        "node_type": "int16",
        "node_id": "int32",
    }
    for c, dt in cast_map.items():
        vals = pd.to_numeric(sample[c], errors="coerce")
        bad = int(vals.isna().sum())
        if bad > 0:
            raise RuntimeError(
                f"sample_submission has non-numeric/NaN values in required column: "
                f"col={c} bad_rows={bad} path={sample_path}"
            )
        sample[c] = vals.astype(dt, copy=False)

    n_rows = len(sample)
    row_id_arr = sample["row_id"].to_numpy(dtype=np.int64, copy=False)
    model_id_arr = sample["model_id"].to_numpy(dtype=np.int16, copy=False)
    event_id_arr = sample["event_id"].to_numpy(dtype=np.int32, copy=False)
    node_type_arr = sample["node_type"].to_numpy(dtype=np.int16, copy=False)
    node_id_arr = sample["node_id"].to_numpy(dtype=np.int32, copy=False)

    pred_out = np.full((n_rows,), np.nan, dtype=np.float32)

    row_id_is_pos = bool(np.array_equal(row_id_arr, np.arange(n_rows, dtype=np.int64)))
    row_id_index = None if row_id_is_pos else pd.Index(row_id_arr)
    log(f"sample rows={n_rows} row_id_is_position={row_id_is_pos}")

    # 3) Dataset metadata for rain features.
    models_root = resolve_models_root(DATASET_ROOT, MODELS_ROOT)
    log(f"models_root={models_root}")

    cv_rows = []
    source_cv_rows = []
    choice_rows = []
    stats_rows = []
    selected_source_by_target: Dict[str, str] = {}
    oof_event_rmse_cache: Dict[Tuple[str, int, int], pd.DataFrame] = {}

    for target in TARGETS:
        model_id, node_type = int(target[0]), int(target[1])
        t0 = time.time()
        log(f"=== Target ({model_id},{node_type}) ===")

        mdir = model_dir(models_root, model_id)
        train_ids = list_event_ids(mdir, "train")
        test_ids = list_event_ids(mdir, "test")
        if not train_ids or not test_ids:
            log(f"skip target ({model_id},{node_type}): train/test events missing")
            continue

        log(
            f"target ({model_id},{node_type}) events train={len(train_ids)} test={len(test_ids)}"
        )

        # 3a) Per-source score for this target: recomputed OOF LB-like score only.
        score_rows = []
        for bi, b in enumerate(bundles, 1):
            score = None
            score_col = "std_rmse"
            score_mode = "oof_recomputed"
            num_events = 0

            cache_key = (b.name, int(target[0]), int(target[1]))
            try:
                st = oof_event_rmse_cache.get(cache_key)
                if st is None:
                    st = aggregate_oof_event_rmse(b.oof_path, b.name, target)
                    oof_event_rmse_cache[cache_key] = st
            except Exception as exc:
                log(f"oof score skip target={target} source={b.name} reason={exc}")
                st = pd.DataFrame()
            if len(st) > 0:
                score = float(st["rmse_std"].mean())
                num_events = int(len(st))
                log(
                    f"oof score target={target} source={b.name} rows={len(st)} "
                    f"std_rmse={score:.6f} ({bi}/{len(bundles)})"
                )

            if score is None or not np.isfinite(score):
                log(f"oof score unavailable target={target} source={b.name}")
                continue

            score_rows.append(
                {
                    "source_name": b.name,
                    "score": float(score),
                    "score_col": str(score_col) if score_col is not None else "",
                    "score_mode": str(score_mode) if score_mode is not None else "",
                    "num_events": int(num_events) if num_events is not None else 0,
                }
            )
            source_cv_rows.append(
                {
                    "model_id": model_id,
                    "node_type": node_type,
                    "source_name": b.name,
                    "mean_std_rmse": float(score),
                    "num_events": int(num_events) if num_events is not None else 0,
                    "score_col": str(score_col) if score_col is not None else "",
                    "score_mode": str(score_mode) if score_mode is not None else "",
                }
            )

        if not score_rows:
            log(f"skip target={target}: no available source score")
            continue

        score_df = pd.DataFrame(score_rows).sort_values(["score", "source_name"]).reset_index(drop=True)
        source_names = score_df["source_name"].astype(str).tolist()
        src_to_idx = {s: i for i, s in enumerate(source_names)}
        fallback_order = [src_to_idx[s] for s in source_names]

        n_classes = int(len(source_names))
        best_single_src = int(fallback_order[0])
        single_score = float(score_df.iloc[0]["score"])
        selected_source_name = source_names[best_single_src]
        selected_source_by_target[f"{model_id}:{node_type}"] = selected_source_name

        top_logs = []
        for i in range(min(5, len(score_df))):
            r = score_df.iloc[i]
            top_logs.append(f"{r['source_name']}:{float(r['score']):.6f}[{r['score_mode']}:{r['score_col']}]")
        log(f"target={target} source ranking top={', '.join(top_logs)}")
        selected_lb_like = np.nan
        selected_lb_like_num_events = 0
        selected_bundle = bundle_by_name.get(selected_source_name)
        if selected_bundle is not None:
            cache_key = (selected_source_name, int(target[0]), int(target[1]))
            try:
                st_selected = oof_event_rmse_cache.get(cache_key)
                if st_selected is None:
                    st_selected = aggregate_oof_event_rmse(selected_bundle.oof_path, selected_source_name, target)
                    oof_event_rmse_cache[cache_key] = st_selected
                if len(st_selected) > 0:
                    selected_lb_like = float(st_selected["rmse_std"].mean())
                    selected_lb_like_num_events = int(len(st_selected))
                    log(
                        f"selector target={target}: single_best={selected_source_name} "
                        f"std_rmse={selected_lb_like:.6f} events={selected_lb_like_num_events}"
                    )
                else:
                    log(f"selector target={target}: single_best={selected_source_name} std_rmse=unavailable events=0")
            except Exception as exc:
                log(f"selector lb-like target={target}: recompute failed reason={exc}")

        cv_rows.append(
            {
                "model_id": model_id,
                "node_type": node_type,
                "num_sources": int(n_classes),
                "num_labeled_events": int(score_df["num_events"].max()) if "num_events" in score_df.columns else 0,
                "selector_cv_std_rmse": float(single_score),
                "single_best_std_rmse": float(single_score),
                "std_rmse": float(selected_lb_like) if np.isfinite(selected_lb_like) else np.nan,
                "num_events": int(selected_lb_like_num_events),
                "oracle_std_rmse": np.nan,
                "selected_source_name": selected_source_name,
                "selected_score_col": str(score_df.iloc[0]["score_col"]),
                "selected_score_mode": str(score_df.iloc[0]["score_mode"]),
            }
        )

        # 3c) Fixed choice for test events (all events in target use same source).
        ev_test_all = np.asarray(test_ids, dtype=np.int32)
        pred_test_src = np.full((len(ev_test_all),), best_single_src, dtype=np.int16)

        # Per-event choice summary.
        test_choice_df = pd.DataFrame(
            {
                "model_id": np.full((len(ev_test_all),), model_id, dtype=np.int16),
                "node_type": np.full((len(ev_test_all),), node_type, dtype=np.int16),
                "event_id": ev_test_all.astype(np.int32, copy=False),
                "chosen_source_idx": pred_test_src.astype(np.int16, copy=False),
            }
        )
        test_choice_df["chosen_source_name"] = [source_names[int(i)] for i in pred_test_src.tolist()]
        choice_rows.append(test_choice_df)

        choice_counts = test_choice_df.groupby("chosen_source_name", as_index=False).agg(selected_events=("event_id", "size"))
        choice_count_map = dict(zip(choice_counts["chosen_source_name"].astype(str), choice_counts["selected_events"].astype(int)))
        n_test_events = max(1, len(ev_test_all))
        choice_log = []
        for sname in source_names:
            cnt = int(choice_count_map.get(sname, 0))
            share = cnt / n_test_events
            choice_log.append(f"{sname}:{cnt}({share:.1%})")
        log(f"target={target} test event selection counts: " + ", ".join(choice_log))

        order = np.argsort(ev_test_all)
        sorted_test_events = ev_test_all[order]
        sorted_test_choice = pred_test_src[order]

        # 3e) Fill submission rows for this target from selected source predictions.
        target_row_idx = np.where((model_id_arr == model_id) & (node_type_arr == node_type))[0]
        if target_row_idx.size == 0:
            log(f"target={target}: no rows in sample_submission")
            continue

        # Precompute (event_id,node_id,occurrence)->sample row mapping for sources without row_id.
        t_ev = event_id_arr[target_row_idx]
        t_nd = node_id_arr[target_row_idx]
        t_key = encode_event_node_key(t_ev, t_nd)
        t_map = pd.DataFrame(
            {
                "key": t_key.astype(np.int64, copy=False),
                "pos": target_row_idx.astype(np.int64, copy=False),
            }
        )
        t_map["occ"] = t_map.groupby("key", sort=False).cumcount().astype(np.int32, copy=False)
        target_key_occ_to_pos = t_map[["key", "occ", "pos"]]
        del t_map

        rows_assigned_main: Dict[str, int] = {s: 0 for s in source_names}
        rows_assigned_fallback: Dict[str, int] = {s: 0 for s in source_names}

        def assign_chunk(
            chunk: pd.DataFrame,
            required_source_idx: int | None,
            only_unresolved: bool,
            no_rowid_occ_counter: Dict[int, int],
        ) -> int:
            if len(chunk) == 0:
                return 0
            pred = chunk["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
            fin = np.isfinite(pred)
            if not np.any(fin):
                return 0

            ev = chunk["event_id"].to_numpy(dtype=np.int32, copy=False)
            choice = map_event_to_choice(ev, sorted_test_events, sorted_test_choice)

            if required_source_idx is not None:
                mask = fin & (choice == int(required_source_idx))
            else:
                mask = fin & (choice >= 0)

            if not np.any(mask):
                return 0

            sub = chunk.loc[mask].copy()
            pred_use = sub["water_level_pred"].to_numpy(dtype=np.float32, copy=False)

            if "row_id" in sub.columns:
                rid = sub["row_id"].to_numpy(dtype=np.int64, copy=False)
                if row_id_is_pos:
                    pos = rid
                else:
                    pos = row_id_index.get_indexer(rid)
                good = (pos >= 0) & (pos < n_rows)
                if not np.any(good):
                    return 0
                pos = pos[good]
                pv = pred_use[good]
            else:
                if "node_id" not in sub.columns:
                    return 0
                skey = encode_event_node_key(
                    sub["event_id"].to_numpy(dtype=np.int32, copy=False),
                    sub["node_id"].to_numpy(dtype=np.int32, copy=False),
                ).astype(np.int64, copy=False)
                s_map = pd.DataFrame(
                    {
                        "key": skey,
                        "pred": pred_use,
                    }
                )
                s_map["occ_local"] = s_map.groupby("key", sort=False).cumcount().astype(np.int32, copy=False)
                key_counts = s_map.groupby("key", sort=False).size().astype(np.int32)
                start_by_key: Dict[int, int] = {}
                for k, c in key_counts.items():
                    kk = int(k)
                    start = int(no_rowid_occ_counter.get(kk, 0))
                    start_by_key[kk] = start
                    no_rowid_occ_counter[kk] = start + int(c)
                s_map["start"] = s_map["key"].map(start_by_key).astype(np.int32, copy=False)
                s_map["occ"] = (s_map["occ_local"] + s_map["start"]).astype(np.int32, copy=False)
                merged = s_map.merge(target_key_occ_to_pos, on=["key", "occ"], how="left", sort=False)
                pos_raw = merged["pos"].to_numpy()
                good = ~pd.isna(pos_raw)
                if not np.any(good):
                    return 0
                pos = pos_raw[good].astype(np.int64, copy=False)
                pv = merged["pred"].to_numpy(dtype=np.float32, copy=False)[good]
            if only_unresolved:
                unresolved = np.isnan(pred_out[pos])
                if not np.any(unresolved):
                    return 0
                pos = pos[unresolved]
                pv = pv[unresolved]
            pred_out[pos] = pv
            return int(len(pos))

        # Main pass: only rows from each event's chosen source.
        for si, sname in enumerate(source_names):
            b = next((x for x in bundles if x.name == sname), None)
            if b is None:
                continue
            rows_assigned = 0
            chunks = 0
            no_rowid_occ_counter: Dict[int, int] = {}
            try:
                for chunk in iter_test_chunks_for_target(b.test_path, target):
                    chunks += 1
                    rows_assigned += assign_chunk(
                        chunk,
                        required_source_idx=si,
                        only_unresolved=False,
                        no_rowid_occ_counter=no_rowid_occ_counter,
                    )
                    if chunks % 10 == 0:
                        log(
                            f"target={target} assign progress source={sname} "
                            f"chunks={chunks} rows_assigned={rows_assigned}"
                        )
            except Exception as exc:
                log(f"target={target} source={sname} assign skipped: {exc}")
                continue
            rows_assigned_main[sname] += int(rows_assigned)
            log(f"target={target} source={sname} assigned rows={rows_assigned}")
            gc.collect()

        unresolved = target_row_idx[np.isnan(pred_out[target_row_idx])]
        if unresolved.size > 0:
            log(f"target={target} unresolved after chosen-source pass: {int(unresolved.size)}")

        # Fallback phase for unresolved rows.
        for si in fallback_order:
            unresolved = target_row_idx[np.isnan(pred_out[target_row_idx])]
            if unresolved.size == 0:
                break
            sname = source_names[int(si)]
            b = next((x for x in bundles if x.name == sname), None)
            if b is None:
                continue
            rows_assigned = 0
            no_rowid_occ_counter: Dict[int, int] = {}
            try:
                for chunk in iter_test_chunks_for_target(b.test_path, target):
                    got = assign_chunk(
                        chunk,
                        required_source_idx=None,
                        only_unresolved=True,
                        no_rowid_occ_counter=no_rowid_occ_counter,
                    )
                    rows_assigned += got
            except Exception:
                continue
            if rows_assigned > 0:
                rows_assigned_fallback[sname] += int(rows_assigned)
                log(f"target={target} fallback source={sname} assigned rows={rows_assigned}")
            gc.collect()

        unresolved = target_row_idx[np.isnan(pred_out[target_row_idx])]
        if unresolved.size > 0:
            resolved = target_row_idx[np.isfinite(pred_out[target_row_idx])]
            fill = float(np.mean(pred_out[resolved])) if resolved.size > 0 else 0.0
            pred_out[unresolved] = np.float32(fill)
            log(
                f"target={target} unresolved rows final fill applied: {int(unresolved.size)} mean={fill:.6f}"
            )

        n_test_events = len(ev_test_all)
        for rank, sname in enumerate(source_names, 1):
            selected_events = int(choice_count_map.get(sname, 0))
            stats_rows.append(
                {
                    "model_id": model_id,
                    "node_type": node_type,
                    "source_name": sname,
                    "source_rank_by_oof": rank,
                    "selected_events": selected_events,
                    "selected_event_share": float(selected_events / n_test_events) if n_test_events > 0 else 0.0,
                    "assigned_rows_main": int(rows_assigned_main.get(sname, 0)),
                    "assigned_rows_fallback": int(rows_assigned_fallback.get(sname, 0)),
                }
            )

        log(f"target={target} done in {time.time() - t0:.1f}s")
        del ev_test_all
        gc.collect()

    # 4) Save outputs
    sub = sample.copy()
    sub["water_level"] = pred_out.astype(np.float32, copy=False)

    missing_all = int(np.sum(~np.isfinite(sub["water_level"].to_numpy(dtype=np.float32, copy=False))))
    if missing_all > 0:
        fill = float(np.nanmean(sub["water_level"].to_numpy(dtype=np.float32, copy=False)))
        sub["water_level"] = sub["water_level"].fillna(fill)
        log(f"global unresolved fill applied rows={missing_all} mean={fill:.6f}")

    out_sub = OUTPUT_DIR / "submission.csv"
    sub.to_csv(out_sub, index=False)
    if PUBLIC_SUBMISSION_PATH:
        Path(PUBLIC_SUBMISSION_PATH).parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(PUBLIC_SUBMISSION_PATH, index=False)

    cv_df = pd.DataFrame(cv_rows)
    overall_lb_like_mean = float(cv_df["std_rmse"].mean()) if (len(cv_df) > 0 and "std_rmse" in cv_df.columns and np.isfinite(cv_df["std_rmse"]).any()) else np.nan
    overall_df = pd.DataFrame([
        {
            "num_targets": int(len(cv_df)),
            "overall_std_rmse": overall_lb_like_mean,
        }
    ])
    cv_path = OUTPUT_DIR / "cv_summary.csv"
    cv_df.to_csv(cv_path, index=False)
    overall_path = OUTPUT_DIR / "overall_summary.csv"
    overall_df.to_csv(overall_path, index=False)
    if np.isfinite(overall_lb_like_mean):
        log(f"overall summary: targets={len(cv_df)} overall_std_rmse={overall_lb_like_mean:.6f}")
    else:
        log(f"overall summary: targets={len(cv_df)} overall_std_rmse=nan")

    src_cv_df = pd.DataFrame(source_cv_rows)
    src_cv_path = OUTPUT_DIR / "source_cv_summary.csv"
    src_cv_df.to_csv(src_cv_path, index=False)

    if choice_rows:
        choice_df = pd.concat(choice_rows, ignore_index=True)
    else:
        choice_df = pd.DataFrame(
            columns=["model_id", "node_type", "event_id", "chosen_source_idx", "chosen_source_name"]
        )

    choice_path = OUTPUT_DIR / "test_event_choices.csv"
    choice_df.to_csv(choice_path, index=False)

    stats_df = pd.DataFrame(stats_rows)
    stats_path = OUTPUT_DIR / "selection_stats.csv"
    stats_df.to_csv(stats_path, index=False)

    meta = {
        "schema_version": SCHEMA_VERSION,
        "mode": "target_singlebest_cv",
        "targets": [f"{m}:{t}" for m, t in TARGETS],
        "bundle_roots": [str(x) for x in BUNDLE_ROOTS],
        "num_bundles": len(bundles),
        "bundle_names": [b.name for b in bundles],
        "selection_rule": "single_best_source_by_target_oof_std_rmse",
        "cv_score_col_priority": CV_SCORE_COL_PRIORITY,
        "selected_source_by_target": selected_source_by_target,
        "overall_std_rmse": overall_lb_like_mean if np.isfinite(overall_lb_like_mean) else None,
        "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "elapsed_sec": float(time.time() - start_all),
    }
    (OUTPUT_DIR / "selector_metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))

    log(f"saved submission: {out_sub}")
    if PUBLIC_SUBMISSION_PATH:
        log(f"saved public submission: {PUBLIC_SUBMISSION_PATH}")
    log(f"saved cv summary: {cv_path}")
    log(f"saved source cv summary: {src_cv_path}")
    log(f"saved overall summary: {overall_path}")
    log(f"saved choice summary: {choice_path}")
    log(f"saved selection stats: {stats_path}")
    log(f"all done in {time.time() - start_all:.1f}s")


if __name__ == "__main__":
    main()


[19:53:10] CONFIG targets=[(1, 1), (1, 2), (2, 1), (2, 2)] bundle_roots=[PosixPath('/kaggle/input'), PosixPath('/kaggle/working/model_bundles'), PosixPath('ensemble/artifacts/model_bundles')] rain_align_mode=min
[19:53:18] bundles detected: 3 names=['physics_informed_gnn_m2_1d_5fold_static_attn_rollout_raingate', 'ridge_lgbm_rain3_rule_single_static_pi_v1__2_2', 'ridge_lgbm_rain3_rule_static_optional__1_1__1_2']
[19:53:57] sample rows=50910192 row_id_is_position=True
[19:53:57] models_root=/kaggle/input/datasets/mtmrs1/urban-flood-bench-re/Models
[19:53:57] === Target (1,1) ===
[19:53:57] target (1,1) events train=68 test=29
[19:53:58] oof score unavailable target=(1, 1) source=physics_informed_gnn_m2_1d_5fold_static_attn_rollout_raingate
[19:54:04] oof score unavailable target=(1, 1) source=ridge_lgbm_rain3_rule_single_static_pi_v1__2_2
[19:54:06] oof agg progress source=ridge_lgbm_rain3_rule_static_optional__1_1__1_2 target=(1, 1) chunks=10 grouped_rows=15
[19:54:08] oof agg progress